# MMVEC plotting with q2 code

Date created: 2/10/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

This script has been adapted by: https://github.com/biocore/mmvec/blob/master/mmvec/q2/_visualizers.py

In [34]:
import os
from os.path import join
import pandas as pd
import numpy as np
import qiime2
import biom
import pkg_resources
import q2templates
import matplotlib.pyplot as plt
from mmvec.heatmap import ranks_heatmap, paired_heatmaps
import seaborn as sns
import warnings

In [35]:
TEMPLATES = pkg_resources.resource_filename('mmvec.q2', 'assets')

In [36]:
# Uncomment group/plot intending to create
skin_group = 'Healthy'
# skin_group = 'Acne_NL'
# skin_group = 'Acne_L'

In [37]:
def _parse_heatmap_metadata_annotations(metadata_column, margin_palette):
    '''
    Transform feature or sample metadata into color vector for annotating
    margin of clustermap.
    Parameters
    ----------
    metadata_column: pd.Series of metadata for annotating plots
    margin_palette: str
        Name of color palette to use for annotating metadata
        along margin(s) of clustermap.
    Returns
    -------
    Returns vector of colors for annotating clustermap and dict mapping colors
    to classes.
    '''
    # Create a categorical palette to identify md col
    metadata_column = metadata_column.astype(str)
    col_names = sorted(metadata_column.unique())

    # Select Color palette
    if margin_palette == 'colorhelix':
        col_palette = sns.cubehelix_palette(
            len(col_names), start=2, rot=3, dark=0.3, light=0.8, reverse=True)
    else:
        col_palette = sns.color_palette(margin_palette, len(col_names))
    class_colors = dict(zip(col_names, col_palette))

    # Convert the palette to vectors that will be drawn on the matrix margin
    col_colors = metadata_column.map(class_colors)

    return col_colors, class_colors


def _parse_taxonomy_strings(taxonomy_series, level):
    '''
    taxonomy_series: pd.Series of semicolon-delimited taxonomy strings
    level: int
        taxonomic level for annotating clustermap.
     Returns
     -------
    Returns a pd.Series of taxonomy names at specified level,
        or terminal annotation
    '''
    return taxonomy_series.apply(lambda x: x.split(';')[:level][-1].strip())


def _process_microbe_metadata(ranks, microbe_metadata, level, margin_palette):
    _warn_metadata_filtering('microbe')
    microbe_metadata, ranks = microbe_metadata.align(
        ranks, join='inner', axis=0)
    # parse semicolon-delimited taxonomy
    if level > -1:
        microbe_metadata = _parse_taxonomy_strings(microbe_metadata, level)
    # map metadata categories to row colors
    row_colors, row_class_colors = _parse_heatmap_metadata_annotations(
        microbe_metadata, margin_palette)

    return microbe_metadata, ranks, row_colors, row_class_colors


def _process_metabolite_metadata(ranks, metabolite_metadata, margin_palette):
    _warn_metadata_filtering('metabolite')
    _ids = set(metabolite_metadata.index) & set(ranks.columns)
    ranks = ranks[sorted(_ids)]
    metabolite_metadata = metabolite_metadata.reindex(ranks.columns)
    # map metadata categories to column colors
    col_colors, col_class_colors = _parse_heatmap_metadata_annotations(
        metabolite_metadata, margin_palette)

    return metabolite_metadata, ranks, col_colors, col_class_colors


def _warn_metadata_filtering(metadata_type):
    warning = ('Conditional probabilities table and {0} metadata will be '
               'filtered to contain only the intersection of IDs in each. If '
               'this behavior is undesired, ensure that all {0} IDs are '
               'present in both the table and the metadata '
               'file'.format(metadata_type))
    warnings.warn(warning, UserWarning)


def _normalize_table(table, method):
    '''
    Normalize column data in a dataframe for plotting in clustermap.

    table: pd.DataFrame
        Input data.
    method: str
        Normalization method to use.

    Returns normalized table as pd.DataFrame
    '''
    if 'col' in method:
        axis = 0
    elif 'row' in method:
        axis = 1
    if 'z_score' in method:
        res = table.apply(lambda x: (x - x.mean()) / x.std(), axis=axis)
    elif 'rel' in method:
        res = table.apply(lambda x: x / x.sum(), axis=axis)
    elif method == 'log10':
        res = table.apply(lambda x: np.log10(x + 1))
    return res.fillna(0)

In [38]:
### Original q2-MMVEC heatmap function
def heatmap(output_dir: str,
            ranks: pd.DataFrame,
            microbe_metadata: pd.Series = None,
            metabolite_metadata: pd.Series = None,
            method: str = 'average',
            metric: str = 'euclidean',
            color_palette: str = 'RdBu_r',
            margin_palette: str = 'cubehelix',
            x_labels: bool = False,
            y_labels: bool = False,
            level: int = -1,
            row_center: bool = True) -> None:
    """
    Generate and save a heatmap without using QIIME's q2templates.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    if microbe_metadata is not None:
        microbe_metadata = microbe_metadata.to_series()
    if metabolite_metadata is not None:
        metabolite_metadata = metabolite_metadata.to_series()
    
    ranks = ranks.T  # Transpose the dataframe

    if row_center:
        ranks = ranks - ranks.mean(axis=0)

    # Generate heatmap
    hotmap = ranks_heatmap(ranks, microbe_metadata, metabolite_metadata,
                           method, metric, color_palette, margin_palette,
                           x_labels, y_labels, level)

    # Save heatmap as PDF and PNG
    hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_original.pdf'), bbox_inches='tight')
    hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_original.png'), bbox_inches='tight')

    print(f"Heatmap saved in {output_dir}")

In [39]:
# Define the output directory
output_dir = f"MMVEC/outputs/{skin_group}/"

# Load the QIIME 2 artifact and extract the dataframe
ranks_artifact = qiime2.Artifact.load(f"MMVEC/outputs/{skin_group}/conditionals.qza")
ranks = ranks_artifact.view(pd.DataFrame)  # Convert to pandas DataFrame
ranks.index.name = None
ranks

,g__Cutibacterium,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Lactobacillus,g__Haemophilus,g__Micrococcus,g__Kocuria,g__Rothia
Isoleucine,2.073708,1.787288,1.289606,1.170427,0.845494,0.839589,0.707319,0.566435,0.560955
Phenylalanine,1.292974,1.066922,0.829983,0.775020,0.697304,0.700995,0.679613,0.644669,0.642208
C19H36O4 Fatty alcohol,0.680918,0.546770,0.417215,0.387365,0.353918,0.356699,0.349892,0.334562,0.333234
Tryptophan,0.656525,0.508254,0.362203,0.328491,0.288243,0.291163,0.282115,0.263772,0.262271
Sorbitane Monooleate,-1.180067,-0.998303,-0.758753,-0.702265,-0.583552,-0.583895,-0.540594,-0.488510,-0.485944
Urocanic acid,-1.699311,-1.405180,-1.034359,-0.947154,-0.774323,-0.775775,-0.714646,-0.638588,-0.634638
N-acyl lipid glutamine-C14:0,-1.824747,-1.505750,-1.105896,-1.011884,-0.827084,-0.828778,-0.763698,-0.682340,-0.678086


In [40]:
# Run the heatmap function with the same parameters as in the qiime command to generate the aesthetically unaltered plots
heatmap(
    output_dir=output_dir,
    ranks=ranks,
    microbe_metadata=None,  # If you have metadata, load it as a pandas Series
    metabolite_metadata=None,  # If you have metabolite metadata, load it
    method="average",
    metric="euclidean",
    color_palette="RdBu_r",
    margin_palette="cubehelix",
    x_labels=True,
    y_labels=True,
    level=50,
    row_center=True
)

Heatmap saved in MMVEC/outputs/Acne_L/


### Create new plots with aesthetic changes

In [41]:
# ### Style-changed q2-MMVEC heatmap function
# def heatmap_stylized(output_dir: str,
#             ranks: pd.DataFrame,
#             microbe_metadata: pd.Series = None,
#             metabolite_metadata: pd.Series = None,
#             method: str = 'average',
#             metric: str = 'euclidean',
#             color_palette: str = 'RdBu_r',
#             margin_palette: str = 'cubehelix',
#             x_labels: bool = False,
#             y_labels: bool = False,
#             row_order=None,
#             col_order=None,
#             level: int = -1,
#             row_center: bool = True) -> None:
#     """
#     Generate and save a heatmap without using QIIME's q2templates.
#     """
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)

#     if microbe_metadata is not None:
#         microbe_metadata = microbe_metadata.to_series()
#     if metabolite_metadata is not None:
#         metabolite_metadata = metabolite_metadata.to_series()
    
#     ranks = ranks.T  # Transpose the dataframe

#     if row_center:
#         ranks = ranks - ranks.mean(axis=0)

#     # **Subset microbe metadata based on rows/columns**
#     if microbe_metadata is not None:
#         microbe_metadata, ranks, row_colors, row_class_colors = _process_microbe_metadata(
#             ranks, microbe_metadata, level, margin_palette)
#     else:
#         row_colors = None

#     # **Subset metabolite metadata based on rows/columns**
#     if metabolite_metadata is not None:
#         metabolite_metadata, ranks, col_colors, col_class_colors = _process_metabolite_metadata(
#             ranks, metabolite_metadata, margin_palette)
#     else:
#         col_colors = None

#     # **Apply custom row & column ordering**
#     if row_order is not None:
#         row_order = [r for r in row_order if r in ranks.index]
#         ranks = ranks.loc[row_order]

#     if col_order is not None:
#         col_order = [c for c in col_order if c in ranks.columns]
#         ranks = ranks[col_order]

#     # Determine the color limits symmetrically around zero
#     vmax = np.max(np.abs(ranks.values))  # Get the maximum absolute value
#     vmin = -vmax  # Ensure symmetry

#     # **Generate heatmap with smaller cells**
#     hotmap = sns.clustermap(ranks, cmap=color_palette,
#                             col_colors=col_colors, row_colors=row_colors,
#                             figsize=(6, 6),  # Reduce figure size
#                             method=method, metric=metric,
#                             dendrogram_ratio=(0.03, 0.03),  # Reduce dendrogram size
#                             row_cluster=False, col_cluster=False,  # Dendrogram clustering
#                             vmin=vmin, vmax=vmax,  # **Ensure 0 is centered**
#                             cbar_kws={'label': 'Log Conditional Probability',
#                                       'orientation': 'horizontal',
#                                       'shrink': 0.5})
    
#     # **Format Colorbar**
#     hotmap.cax.set_position([0.25, -0.1, 0.5, 0.02])  # Move colorbar down
#     cbar = hotmap.ax_heatmap.collections[0].colorbar
#     cbar.set_label("Log Conditional Probability", fontsize=10, labelpad=10)
#     cbar.ax.tick_params(labelsize=10)

#     # ** Add title**
#     titles = {
#         'Acne_L': 'Acne Lesional',
#         'Acne_NL': 'Acne Non-Lesional',
#         'Healthy': 'Healthy'
#     }
#     plt.suptitle(titles.get(skin_group, ''), fontsize=16, y=1.0)

#     # **Increase Axis Label Sizes**
#     hotmap.ax_heatmap.tick_params(axis='x', labelsize=12, rotation=90)
#     hotmap.ax_heatmap.tick_params(axis='y', labelsize=12)

#     # Save heatmap as PNG and SVG
#     hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_reformatted.png'), bbox_inches='tight', dpi=600)
#     hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_reformatted.svg'), bbox_inches='tight')

#     print(f"Heatmap saved in {output_dir}")


In [ ]:
def heatmap_stylized(output_dir: str,
            ranks: pd.DataFrame,
            microbe_metadata: pd.Series = None,
            metabolite_metadata: pd.Series = None,
            method: str = 'average',
            metric: str = 'euclidean',
            color_palette: str = 'RdBu_r',
            margin_palette: str = 'cubehelix',
            x_labels: bool = False,
            y_labels: bool = False,
            row_order=None,
            col_order=None,
            level: int = -1,
            row_center: bool = True) -> None:
    """
    Generate and save a heatmap without using QIIME's q2templates.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    if microbe_metadata is not None:
        microbe_metadata = microbe_metadata.to_series()
    if metabolite_metadata is not None:
        metabolite_metadata = metabolite_metadata.to_series()
    
    ranks = ranks.T  # Transpose the dataframe

    if row_center:
        ranks = ranks - ranks.mean(axis=0)

    # Subset microbe metadata based on rows/columns
    if microbe_metadata is not None:
        microbe_metadata, ranks, row_colors, row_class_colors = _process_microbe_metadata(
            ranks, microbe_metadata, level, margin_palette)
    else:
        row_colors = None

    if metabolite_metadata is not None:
        metabolite_metadata, ranks, col_colors, col_class_colors = _process_metabolite_metadata(
            ranks, metabolite_metadata, margin_palette)
    else:
        col_colors = None

    if row_order is not None:
        row_order = [r for r in row_order if r in ranks.index]
        ranks = ranks.loc[row_order]

    if col_order is not None:
        col_order = [c for c in col_order if c in ranks.columns]
        ranks = ranks[col_order]

    # Determine color limits symmetrically around zero
    vmax = np.max(np.abs(ranks.values))
    vmin = -vmax

    # Generate heatmap
    hotmap = sns.clustermap(ranks, cmap=color_palette,
                            col_colors=col_colors, row_colors=row_colors,
                            figsize=(6, 6),
                            method=method, metric=metric,
                            dendrogram_ratio=(0.03, 0.03),
                            row_cluster=False, col_cluster=False,
                            vmin=vmin, vmax=vmax,
                            cbar_kws={'label': 'Log Conditional Probability',
                                      'orientation': 'horizontal',
                                      'shrink': 0.5})

    # Annotate each cell with co-occurrence value (2 decimal places)
    for i in range(ranks.shape[0]):
        for j in range(ranks.shape[1]):
            value = ranks.iloc[i, j]
            hotmap.ax_heatmap.text(j + 0.5, i + 0.5, f"{value:.2f}",
                                   ha='center', va='center',
                                   color='black', fontsize=8)

    # Format colorbar
    hotmap.cax.set_position([0.25, -0.1, 0.5, 0.02])
    cbar = hotmap.ax_heatmap.collections[0].colorbar
    cbar.set_label("Log Conditional Probability", fontsize=10, labelpad=10)
    cbar.ax.tick_params(labelsize=10)

    # Add title
    titles = {
        'Acne_L': 'Acne Lesional',
        'Acne_NL': 'Acne Non-Lesional',
        'Healthy': 'Healthy'
    }
    plt.suptitle(titles.get(skin_group, ''), fontsize=16, y=1.0)

    # Increase axis label sizes
    hotmap.ax_heatmap.tick_params(axis='x', labelsize=12, rotation=90)
    hotmap.ax_heatmap.tick_params(axis='y', labelsize=12)

    # Save as PNG and SVG
    hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_reformatted.png'), bbox_inches='tight', dpi=600)
    hotmap.savefig(join(output_dir, f'{skin_group}_MMVEC-heatmap_reformatted.svg'), bbox_inches='tight')

    print(f"Heatmap saved in {output_dir}")


In [42]:
# Define the output directory
output_dir = f"MMVEC/outputs/{skin_group}/"

# Ensure row and column orders are the same across all figures
microbe_order = [' g__Cutibacterium', ' g__Staphylococcus', ' g__Micrococcus', ' g__Haemophilus', ' g__Kocuria', ' g__Lactobacillus', ' g__Rothia', ' g__Streptococcus', ' g__Lawsonella', ' g__Corynebacterium']

metabolite_order = ['Tryptophan', 'Phenylalanine', 'C19H36O4 Fatty alcohol', 
                    'Urocanic acid', 'Sorbitane Monooleate', 'Glutamine-C14:0']

# Load the QIIME 2 artifact and extract the dataframe
ranks_artifact = qiime2.Artifact.load(f"MMVEC/outputs/{skin_group}/conditionals.qza")
ranks = ranks_artifact.view(pd.DataFrame)  # Convert to pandas DataFrame
ranks.index.name = None

# Reorder by microbe_order and metabolite_order
ranks = ranks.reindex(index=metabolite_order, columns=microbe_order)
# ranks = ranks.iloc[g.dendrogram_row.reordered_ind, g.dendrogram_col.reordered_ind]


ranks

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Haemophilus,g__Kocuria,g__Lactobacillus,g__Rothia
Tryptophan,0.656525,0.508254,0.328491,0.362203,0.282115,0.291163,0.263772,0.288243,0.262271
Phenylalanine,1.292974,1.066922,0.775020,0.829983,0.679613,0.700995,0.644669,0.697304,0.642208
C19H36O4 Fatty alcohol,0.680918,0.546770,0.387365,0.417215,0.349892,0.356699,0.334562,0.353918,0.333234
Urocanic acid,-1.699311,-1.405180,-0.947154,-1.034359,-0.714646,-0.775775,-0.638588,-0.774323,-0.634638
Sorbitane Monooleate,-1.180067,-0.998303,-0.702265,-0.758753,-0.540594,-0.583895,-0.488510,-0.583552,-0.485944
N-acyl lipid glutamine-C14:0,-1.824747,-1.505750,-1.011884,-1.105896,-0.763698,-0.828778,-0.682340,-0.827084,-0.678086


In [43]:
# If trying to do explicit probabilities, see https://github.com/biocore/mmvec/issues/93
# from skbio.stats.composition import clr_inv

# Obtain the explicit probabilities from these ranks
# probs = ranks.apply(clr_inv, axis=0)

# Check column sums (should equal 1 after applying softmax transform)
# col_sums = np.sum(probs, axis=0)
# print("Row sums: ", col_sums)

In [44]:
# Run the heatmap function with to generate same MMVEC heatmap but with preferred aesthetics
heatmap_stylized(
    output_dir=output_dir,
    ranks=ranks,
    microbe_metadata=None,  # If you have metadata, load it as a pandas Series
    metabolite_metadata=None,  # If you have metabolite metadata, load it
    method="average",
    metric="euclidean",
    color_palette="RdBu_r",
    margin_palette="cubehelix",
    x_labels=True,
    y_labels=True,
    row_order=microbe_order,
    col_order=metabolite_order,
    level=50,
    row_center=True
)

Heatmap saved in MMVEC/outputs/Acne_L/
